# Same Yardstick: measuring trench movement identically at the Combine and after the snap

**NFL Big Data Bowl 2027 — companion notebook for the writeup.** Everything in the writeup (numbers, models, figures) is produced by this notebook from the competition files only.

Pipeline:
1. One feature function (`trench_features.py`) measures depth, lateral movement, speed and timing in the first 1.5 s of movement — applied to Combine drill attempts (`PASS_RUSH_DROP` for OL, `PASS_RUSH_DRILL` for DL, plus the 40) and to every regular-season pass-pro / pass-rush snap of the 239 OL/DL prospects, starting at `ball_snap`.
2. Reliability first (attempt 1 vs 2 at the Combine; odd vs even weeks in games).
3. Transfer: same metric, Combine → snaps, with height, weight and draft slot held fixed (bootstrap CIs).
4. Outcome: in-game mechanic → pressure (DL) / pressure allowed (OL), with situational controls and within-player (fixed-effect) models.

Runs in ~3 minutes on CPU.

In [ ]:
import glob, os, sys, shutil
from pathlib import Path
BASE = os.path.dirname(glob.glob("/kaggle/input/**/games.csv", recursive=True)[0]) + "/"
ROOT = Path("/kaggle/working"); (ROOT / "src").mkdir(exist_ok=True); (ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
if not os.path.lexists(ROOT / "data" / "raw"):
    os.symlink(BASE.rstrip("/"), ROOT / "data" / "raw")
sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
print("BASE", BASE)

## Source modules (also at the GitHub repo linked in the writeup)

In [ ]:
Path('src/trench_features.py').write_text(r'''"""Shared kinematic features for trench (OL/DL) movement, used for BOTH
Combine drill attempts and in-game snaps so the two sides are measured
identically.

Input frame table columns (polars):
  key cols..., t (sec, 0 = movement start / ball snap), x, y, s, a,
  ux, uy  (unit axis vector: direction of "depth" = toward the offensive
           backfield for in-game, net displacement direction for a drill),
  o       (orientation deg, optional; NaN if absent),
  face    (target facing deg, optional).

NOTE: the Combine file's `dis` column is not a per-frame step (it behaves
like a squared step), so nothing here uses `dis`; distance comes from x/y.
"""
import polars as pl

WINDOWS = (0.5, 1.0, 1.5, 2.0)
TOL = 0.051  # half a frame at 10 Hz


def _at(col: str, tt: float, name: str | None = None) -> pl.Expr:
    """Value of `col` at the frame closest to time tt (within one frame)."""
    name = name or f"{col}_{int(round(tt * 10)):02d}"
    return pl.col(col).filter((pl.col("t") - tt).abs() < TOL).first().alias(name)


def _in(lo: float, hi: float) -> pl.Expr:
    return (pl.col("t") >= lo) & (pl.col("t") <= hi)


def add_depth_lateral(frames: pl.DataFrame, key: list[str], ref: str = "t0") -> pl.DataFrame:
    """Add depth (along axis) and signed lateral (perp. to axis) relative to
    the reference position of each group: the frame at t==0 (in-game, the
    snap) or the first recorded frame (Combine, ref="first"). If a
    `depth_shift` column exists (Combine onset back-extrapolation) it is
    added to depth."""
    if ref == "first":
        x0 = pl.col("x").sort_by("t").first().over(key); y0 = pl.col("y").sort_by("t").first().over(key)
    else:
        x0 = pl.col("x").filter(pl.col("t").abs() < TOL).first().over(key)
        y0 = pl.col("y").filter(pl.col("t").abs() < TOL).first().over(key)
    dx, dy = pl.col("x") - x0, pl.col("y") - y0
    shift = pl.col("depth_shift") if "depth_shift" in frames.columns else pl.lit(0.0)
    out = frames.with_columns(
        (dx * pl.col("ux") + dy * pl.col("uy") + shift).alias("depth"),
        (-dx * pl.col("uy") + dy * pl.col("ux")).alias("lat"),
    )
    if "o" in out.columns and "face" in out.columns:
        rel = ((pl.col("o") - pl.col("face") + 180.0) % 360.0 - 180.0).abs()
        out = out.with_columns(rel.alias("orel"))
    else:
        out = out.with_columns(pl.lit(None, dtype=pl.Float64).alias("orel"))
    return out


def window_features(frames: pl.DataFrame, key: list[str]) -> pl.DataFrame:
    """One row per group with fixed-window kinematic features."""
    aggs = []
    for tt in WINDOWS:
        aggs += [_at("depth", tt), _at("lat", tt)]
    aggs += [
        _at("s", 0.5), _at("s", 1.0), _at("s", 1.5),
        pl.col("s").filter(_in(0.0, 1.5)).max().alias("smax_15"),
        pl.col("s").filter(_in(0.0, 1.0)).max().alias("smax_10"),
        pl.col("a").filter(_in(0.0, 0.5)).max().alias("amax_05"),
        pl.col("a").filter(_in(0.0, 1.0)).max().alias("amax_10"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("s") >= 1.0)).min().alias("t_s1"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("s") >= 2.0)).min().alias("t_s2"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("s") >= 3.0)).min().alias("t_s3"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("depth") >= 1.0)).min().alias("t_d1"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("depth") >= 2.0)).min().alias("t_d2"),
        pl.col("t").filter((pl.col("t") >= 0) & (pl.col("depth") >= 3.0)).min().alias("t_d3"),
        # onset-free quantities: speed when depth first reaches 1 / 2 yd
        pl.col("s").filter((pl.col("t") >= 0) & (pl.col("depth") >= 1.0)).first().alias("s_at_d1"),
        pl.col("s").filter((pl.col("t") >= 0) & (pl.col("depth") >= 2.0)).first().alias("s_at_d2"),
        _at("orel", 0.0, "orel_00"), _at("orel", 1.0, "orel_10"), _at("orel", 0.5, "orel_05"),
        _at("s", -0.2, "pre_s"),
        pl.len().alias("n_frames"),
        pl.col("t").min().alias("t_min"), pl.col("t").max().alias("t_max"),
    ]
    out = frames.group_by(key, maintain_order=True).agg(aggs)
    # set angle: 0 = straight back/forward along axis, 90 = pure lateral
    return out.with_columns(
        (pl.arctan2(pl.col("lat_10").abs(), pl.col("depth_10")).degrees()).alias("set_angle_10"),
        pl.col("lat_10").abs().alias("lat_abs_10"),
        pl.col("lat_15").abs().alias("lat_abs_15"),
        (pl.col("t_d2") - pl.col("t_d1")).alias("t_d1_d2"),
        # side of the set: lat < 0 is the offensive left in both files (an OL's own
        # left; a DL's own right)
        pl.when(pl.col("lat_10") < 0).then(pl.lit("L")).otherwise(pl.lit("R")).alias("side_10"),
    )


def speed_profiles(frames: pl.DataFrame, key: list[str], t_hi: float = 2.0) -> pl.DataFrame:
    """Per group: list columns of t, s, depth, lat, orel for 0 <= t <= t_hi
    (10 Hz), so curves can be compared later without re-reading tracking."""
    f = frames.filter((pl.col("t") >= -TOL) & (pl.col("t") <= t_hi + TOL)).sort(key + ["t"])
    return f.group_by(key, maintain_order=True).agg(
        pl.col("t").round(1).alias("t_list"), pl.col("s").alias("s_list"),
        pl.col("depth").alias("depth_list"), pl.col("lat").alias("lat_list"),
        pl.col("orel").alias("orel_list"),
    )


def extract(frames: pl.DataFrame, key: list[str]) -> pl.DataFrame:
    return window_features(add_depth_lateral(frames, key), key)
''', encoding='utf-8')
print('wrote src/trench_features.py')

In [ ]:
Path('src/build_combine_trench_metrics.py').write_text(r'''"""Build per-attempt Combine kinematic features for trench drills.

Usage: python src/build_combine_trench_metrics.py
Writes data/processed/combine_trench_attempts.csv
"""
from pathlib import Path
import sys

import numpy as np
import polars as pl

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from trench_features import add_depth_lateral, speed_profiles, window_features  # noqa: E402

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

# drill name normalisation (2023 vs 2024-25 spellings)
DRILL_ALIAS = {"PASS_PRO-MIRROR_DRILL": "PASS_PRO_MIRROR_DRILL", "LINE": "LINE_DRILL"}
DRILLS = [
    "PASS_RUSH_DROP", "PASS_PRO_MIRROR_DRILL", "FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE",
    "PASS_RUSH_DRILL", "RUN_THE_HOOP_DRILL", "FORTY_YARD_DASH",
]
START_SPEED = 0.5  # yd/s: movement onset defines t = 0 for a drill attempt
# The Combine file has no start signal and most attempts are already moving
# on their first recorded frame. Onset is back-extrapolated assuming constant
# acceleration from rest over the first frames: t_shift = v0/a0, d_shift =
# v0^2/(2 a0). Attempts whose first frame is already fast are excluded.
A0_FLOOR = 1.5      # yd/s^2 floor for the extrapolation acceleration
MAX_FIRST_SPEED = 2.0  # yd/s: first recorded frame faster than this -> drop attempt


def hitch_features(df: pl.DataFrame) -> pl.DataFrame:
    """OL PASS_RUSH_DROP has a kick-slide phase, a hitch (speed dip), then a
    sprint. Find first speed peak (>1 yd/s), the following dip, and the time
    from dip to 5 yd/s (redirect burst)."""
    rows = []
    for (ev,), g in df.group_by("event_id", maintain_order=True):
        t = g["t"].to_numpy(); s = g["s"].to_numpy(); d = g["depth"].to_numpy()
        m = t >= 0
        t, s, d = t[m], s[m], d[m]
        rec = {"event_id": ev, "peak1_t": np.nan, "peak1_s": np.nan, "dip_t": np.nan,
               "dip_s": np.nan, "dip_depth": np.nan, "redirect_t": np.nan}
        if len(s) < 8:
            rows.append(rec); continue
        # smooth 3 frames
        ss = np.convolve(s, np.ones(3) / 3, mode="same")
        peaks = [i for i in range(1, len(ss) - 1) if ss[i] >= ss[i - 1] and ss[i] > ss[i + 1] and ss[i] > 1.0]
        if not peaks:
            rows.append(rec); continue
        p = peaks[0]
        rec["peak1_t"], rec["peak1_s"] = t[p], ss[p]
        dips = [i for i in range(p + 1, len(ss) - 1) if ss[i] <= ss[i - 1] and ss[i] < ss[i + 1]]
        if dips:
            q = dips[0]
            rec["dip_t"], rec["dip_s"], rec["dip_depth"] = t[q], ss[q], d[q]
            after = np.where((np.arange(len(ss)) > q) & (ss >= 5.0))[0]
            if len(after):
                rec["redirect_t"] = t[after[0]] - t[q]
        rows.append(rec)
    return pl.DataFrame(rows)


def main() -> None:
    ct = pl.read_csv(RAW / "combine_tracking.csv", schema_overrides={"time": pl.String})
    ct = ct.filter(pl.col("entity_type") == "PLAYER").with_columns(
        pl.col("drill_name").replace(DRILL_ALIAS).alias("drill_name")
    ).filter(pl.col("drill_name").is_in(DRILLS))
    ct = ct.with_columns(pl.col("time").str.to_datetime("%Y-%m-%dT%H:%M:%S%.f").alias("ts"))
    ct = ct.sort(["event_id", "ts"])

    # movement onset per attempt: first frame with s >= START_SPEED, then
    # back-extrapolated to rest (see A0_FLOOR); frames before onset dropped
    key = ["event_id"]
    onset = pl.col("ts").filter(pl.col("s") >= START_SPEED).first().over(key)
    ct = ct.with_columns(((pl.col("ts") - onset).dt.total_milliseconds() / 1000.0).alias("t"),
                         pl.col("s").sort_by("ts").first().over(key).alias("first_recorded_speed"))
    ct = ct.filter(pl.col("t").is_not_null() & (pl.col("t") >= 0))
    v0 = pl.col("s").sort_by("t").first().over(key)
    a0 = pl.max_horizontal(pl.col("a").sort_by("t").head(3).mean().over(key), pl.lit(A0_FLOOR))
    ct = ct.with_columns(v0.alias("first_speed"), (v0 / a0).alias("t_shift"), (v0 ** 2 / (2 * a0)).alias("depth_shift"))
    ct = ct.filter(pl.col("first_speed") < MAX_FIRST_SPEED).with_columns((pl.col("t") + pl.col("t_shift")).alias("t"))

    # axis: the two primary drills are run along the field x axis from a stance
    # facing +x (OL kick-slides to -x, DL bursts to +x), so use that axis so
    # that depth/lat mean the same thing as in-game (lat < 0 = player's left).
    # Other drills: unit vector of net displacement from onset to last frame.
    first_x = pl.col("x").sort_by("t").first().over(key); first_y = pl.col("y").sort_by("t").first().over(key)
    last_x = pl.col("x").sort_by("t").last().over(key); last_y = pl.col("y").sort_by("t").last().over(key)
    ct = ct.with_columns((last_x - first_x).alias("_dx"), (last_y - first_y).alias("_dy"))
    norm = pl.max_horizontal((pl.col("_dx") ** 2 + pl.col("_dy") ** 2).sqrt(), pl.lit(1e-6))
    ct = ct.with_columns(
        pl.when(pl.col("drill_name") == "PASS_RUSH_DROP").then(-1.0)
        .when(pl.col("drill_name") == "PASS_RUSH_DRILL").then(1.0)
        .otherwise(pl.col("_dx") / norm).alias("ux"),
        pl.when(pl.col("drill_name").is_in(["PASS_RUSH_DROP", "PASS_RUSH_DRILL"])).then(0.0)
        .otherwise(pl.col("_dy") / norm).alias("uy"),
    )

    meta = ct.group_by(key, maintain_order=True).agg(
        pl.col("nfl_id").first(), pl.col("draft_year").first(), pl.col("drill_type").first(),
        pl.col("drill_name").first(), pl.col("attempt").first(),
        norm.first().alias("net_disp"), pl.col("s").max().alias("smax_all"),
        pl.col("t").max().alias("duration"), pl.col("first_speed").first(), pl.col("t_shift").first(),
        pl.col("first_recorded_speed").first(),
    )
    fr = add_depth_lateral(ct, key, ref="first")
    feats = window_features(fr, key)
    hitch = hitch_features(fr.filter(pl.col("drill_name") == "PASS_RUSH_DROP"))
    out = meta.join(feats, on=key).join(hitch, on=key, how="left")
    out.write_csv(OUT / "combine_trench_attempts.csv")
    speed_profiles(fr, key).join(meta, on=key).write_parquet(OUT / "combine_trench_profiles.parquet")
    print(out.shape)
    print(out.group_by("drill_name").agg(pl.len(), pl.col("nfl_id").n_unique(), pl.col("first_recorded_speed").median(),
                                         (pl.col("first_recorded_speed") >= START_SPEED).mean().alias("frac_moving_at_first_frame"),
                                         pl.col("t_shift").median()).sort("drill_name"))
    cols = ["depth_05", "depth_10", "depth_15", "s_05", "s_10", "smax_15", "amax_05", "t_s2", "t_d1", "set_angle_10", "peak1_s", "dip_t", "redirect_t"]
    for dn in ["PASS_RUSH_DROP", "PASS_RUSH_DRILL"]:
        print(dn)
        print(out.filter(pl.col("drill_name") == dn).select(cols).describe())


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('wrote src/build_combine_trench_metrics.py')

In [ ]:
Path('src/trench_analysis.py').write_text(r'''"""Analysis helpers for the Combine -> in-game -> outcome chain (trench).

All functions are pure pandas/numpy so they run locally and inside the final
Kaggle notebook unchanged.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

OL_POS = ["T", "G", "C"]
DL_POS = ["DT", "DE", "OLB", "NT"]

GAME_METRICS = ["depth_05", "depth_10", "depth_15", "lat_abs_10", "s_05", "s_10", "smax_10", "smax_15",
                "amax_05", "t_s2", "t_d1", "t_d2", "t_d1_d2", "s_at_d1", "set_angle_10", "orel_00",
                "orel_05", "orel_10"]


# ----------------------------------------------------------------------------
# loading / cleaning
# ----------------------------------------------------------------------------
def load_game_features(path) -> pd.DataFrame:
    g = pd.read_csv(path, low_memory=False)
    g = g[(g.play_nullified_by_penalty != "Y") & (g.quarter <= 4)]
    # sanity: player should be still before the snap; drop mis-tagged snaps
    g = g[(g.pre_s.isna()) | (g.pre_s < 1.5)]
    g = g[g.depth_10.notna() & (g.t_max >= 1.5)]  # full 1.5 s window recorded
    g["nfl_position"] = g.nfl_position.replace({"NT": "DT"})  # NT n=3: pool with interior
    g["pressure_allowed"] = g.pressure_allowed.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})
    g["pressured"] = g.time_to_pressure.notna().astype("float64")  # DL generated pressure
    g["quick_pressure"] = g.quick_pressure.fillna(0.0)
    g["side_game"] = np.where(g.lat_10 < 0, "L", "R")
    num = GAME_METRICS + ["pressure_allowed", "peak_pressure_probability_allowed", "pressured",
                          "quick_pressure", "player_get_off", "sack", "sack_allowed", "dropback_duration",
                          "pass_rushers_encountered", "time_to_pressure", "lat_10"]
    for c in num:
        g[c] = pd.to_numeric(g[c], errors="coerce").astype("float64")
    return g


# ----------------------------------------------------------------------------
# per-player aggregation and reliability
# ----------------------------------------------------------------------------
def player_means(g: pd.DataFrame, by: list[str], metrics: list[str], min_plays: int = 50) -> pd.DataFrame:
    agg = g.groupby(by)[metrics].mean()
    agg["n_plays"] = g.groupby(by).size()
    return agg[agg.n_plays >= min_plays].reset_index()


def split_half_reliability(g: pd.DataFrame, by: list[str], metrics: list[str], min_plays: int = 50) -> pd.DataFrame:
    """Odd vs even weeks, Spearman-Brown corrected, per metric."""
    odd = g[g.week % 2 == 1]; even = g[g.week % 2 == 0]
    a = player_means(odd, by, metrics, min_plays // 2).set_index(by)
    b = player_means(even, by, metrics, min_plays // 2).set_index(by)
    common = a.index.intersection(b.index)
    rows = []
    for m in metrics:
        x, y = a.loc[common, m].astype("float64"), b.loc[common, m].astype("float64")
        ok = x.notna() & y.notna()
        r = np.corrcoef(x[ok].values, y[ok].values)[0, 1] if ok.sum() > 5 else np.nan
        rows.append({"metric": m, "n": int(ok.sum()), "r_half": r, "r_sb": 2 * r / (1 + r) if r == r else np.nan})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# correlation with uncertainty / confounder control
# ----------------------------------------------------------------------------
def fisher_ci(r: float, n: int, alpha: float = 0.05) -> tuple[float, float]:
    if n < 4 or not np.isfinite(r):
        return (np.nan, np.nan)
    z = np.arctanh(np.clip(r, -0.999, 0.999)); se = 1 / np.sqrt(n - 3)
    from scipy.stats import norm
    q = norm.ppf(1 - alpha / 2)
    return (float(np.tanh(z - q * se)), float(np.tanh(z + q * se)))


def residualize(df: pd.DataFrame, cols: list[str], controls: list[str]) -> pd.DataFrame:
    """OLS-residualize each of `cols` on `controls` (with intercept)."""
    out = df.copy()
    X = np.column_stack([np.ones(len(df))] + [df[c].astype(float).values for c in controls])
    for c in cols:
        y = df[c].astype(float).values
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        out[c] = y - X @ beta
    return out


def _partial_r(x: np.ndarray, y: np.ndarray, X: np.ndarray) -> float:
    """Pearson r between the residuals of x and y on the design matrix X."""
    bx, *_ = np.linalg.lstsq(X, x, rcond=None); by, *_ = np.linalg.lstsq(X, y, rcond=None)
    rx, ry = x - X @ bx, y - X @ by
    den = np.sqrt((rx ** 2).sum() * (ry ** 2).sum())
    return float((rx * ry).sum() / den) if den > 0 else np.nan


def corr_table(df: pd.DataFrame, xs: list[str], ys: list[str], controls: list[str] | None = None,
               n_boot: int = 2000, seed: int = 0) -> pd.DataFrame:
    """Pearson r for every x-y pair, bootstrap 95% CI over rows, optional
    partial correlation (both sides residualized on controls)."""
    rng = np.random.default_rng(seed)
    rows = []
    for x in xs:
        for y in ys:
            ctrl = [c for c in (controls or []) if c not in (x, y)]  # never control for x or y itself
            cols = list(dict.fromkeys([x, y] + ctrl))
            d = df[cols].dropna()
            n = len(d)
            if n < 8:
                rows.append({"x": x, "y": y, "n": n, "r": np.nan, "lo": np.nan, "hi": np.nan}); continue
            xv, yv = d[x].astype(float).values, d[y].astype(float).values
            X = np.column_stack([np.ones(n)] + [d[c].astype(float).values for c in ctrl])
            r = _partial_r(xv, yv, X)
            br = np.empty(n_boot)
            for b in range(n_boot):  # re-fit the control regression inside every resample
                idx = rng.integers(0, n, size=n)
                br[b] = _partial_r(xv[idx], yv[idx], X[idx])
            lo, hi = np.nanpercentile(br, [2.5, 97.5])
            rows.append({"x": x, "y": y, "n": n, "r": r, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)


def zscore_within(df: pd.DataFrame, cols: list[str], by: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        g = out.groupby(by)[c]
        out[c] = (out[c] - g.transform("mean")) / g.transform("std")
    return out
''', encoding='utf-8')
print('wrote src/trench_analysis.py')

In [ ]:
Path('src/make_figures.py').write_text(r'''"""Compute the writeup numbers and draw the figures.

Runs locally (paths under data/processed) and inside the Kaggle notebook
(paths under /kaggle/working) - set PROC / OUT accordingly.

Usage: python src/make_figures.py
"""
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from trench_analysis import (GAME_METRICS, corr_table, load_game_features, player_means,  # noqa: E402
                             split_half_reliability, zscore_within)

PROC = ROOT / "data" / "processed"
RAW = ROOT / "data" / "raw"
OUT = ROOT / "output" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

C_COMBINE, C_GAME, C_OL, C_DL = "#d95f02", "#1b9e77", "#2c7fb8", "#c51b7d"
LABELS = {"depth_10": "depth at 1.0 s (yd)", "s_10": "speed at 1.0 s", "t_d1": "time to 1 yd", "s_at_d1": "speed at 1 yd",
          "t_d1_d2": "time 1→2 yd", "smax_15": "peak speed 0–1.5 s", "set_angle_10": "set angle at 1.0 s", "orel_10": "hip angle at 1.0 s",
          "lat_abs_10": "lateral at 1.0 s", "player_get_off": "NGS get-off", "pressure_allowed": "pressure allowed",
          "peak_pressure_probability_allowed": "peak pressure prob. allowed", "pressured": "pressure rate", "depth_05": "depth at 0.5 s",
          "s_05": "speed at 0.5 s", "amax_05": "peak accel 0–0.5 s"}
MET = ["depth_10", "s_10", "t_d1", "s_at_d1", "t_d1_d2", "smax_15", "set_angle_10", "lat_abs_10"]
CTRL = ["combine_height", "combine_weight", "pick"]
RES: dict = {}


def load_all():
    g = load_game_features(PROC / "trench_game_features.csv")
    att = pd.read_csv(PROC / "combine_trench_attempts.csv")
    players = pd.read_csv(RAW / "players.csv")
    comb = pd.read_csv(RAW / "combine_results.csv")
    base = players[["nfl_id", "display_name", "nfl_position", "draft_year", "draft_overall_pick"]].merge(
        comb[["nfl_id", "combine_height", "combine_weight", "forty", "ten_yd_split"]], on="nfl_id")
    base["pick"] = base.draft_overall_pick.fillna(270)
    return g, att, players, base


def combine_player(att, drill, cols, ids=None):
    """Mean over attempts of year-centred (z within draft_year) metrics; `ids`
    restricts the centring population (e.g. linemen only for the 40)."""
    d = att[att.drill_name == drill].copy()
    if ids is not None:
        d = d[d.nfl_id.isin(ids)]
    for c in cols:
        grp = d.groupby("draft_year")[c]
        d[c] = (d[c] - grp.transform("mean")) / grp.transform("std")
    return d.groupby("nfl_id")[cols].mean().reset_index()


def player_table(g, att, base, side, drill, outs, min_plays=100):
    """One row per player: Combine drill z-metrics (c_), 40 z-metrics (f_), in-game means (g_) z within position."""
    gm = player_means(g[g.side == side], ["nfl_id"], GAME_METRICS + outs, min_plays)
    gm = gm.rename(columns={m: "g_" + m for m in GAME_METRICS + outs})
    ids = base[base.nfl_position.isin(["T", "G", "C"] if side == "OL" else ["DT", "DE", "OLB", "NT"])].nfl_id
    c = combine_player(att, drill, MET).rename(columns={m: "c_" + m for m in MET})
    f = combine_player(att, "FORTY_YARD_DASH", MET, ids).rename(columns={m: "f_" + m for m in MET})
    d = base.merge(gm, on="nfl_id").merge(c, on="nfl_id", how="left").merge(f, on="nfl_id", how="left")
    return zscore_within(d, [x for x in d.columns if x.startswith("g_")], ["nfl_position"])


# ----------------------------------------------------------------------------- figures
def fig1_profiles():
    """Mean speed profile 0-1.5 s: Combine drill vs in-game, OL and DL."""
    cp = pd.read_parquet(PROC / "combine_trench_profiles.parquet")
    gp = pd.read_parquet(PROC / "trench_game_profiles.parquet")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, side, drill, title in [(axes[0], "OL", "PASS_RUSH_DROP", "Offensive line: kick-slide drill vs pass-pro snap"),
                                   (axes[1], "DL", "PASS_RUSH_DRILL", "Defensive line: pass-rush drill vs pass-rush snap")]:
        for df, lab, col in [(cp[cp.drill_name == drill], "Combine drill", C_COMBINE), (gp[gp.side == side], "NFL snaps (REG)", C_GAME)]:
            t = np.round(np.concatenate(df.t_list.values), 1); s = np.concatenate(df.s_list.values)
            ok = (t >= 0) & (t <= 1.5)
            prof = pd.DataFrame({"t": t[ok], "s": s[ok]}).groupby("t").s.agg(["mean", lambda v: v.quantile(.25), lambda v: v.quantile(.75)])
            ax.plot(prof.index, prof["mean"], color=col, lw=2.2, label=f"{lab} (n={len(df):,})")
            ax.fill_between(prof.index, prof.iloc[:, 1], prof.iloc[:, 2], color=col, alpha=.15)
        ax.set_title(title, fontsize=10); ax.set_xlabel("seconds after movement start / ball snap"); ax.grid(alpha=.3); ax.legend(fontsize=8, loc="upper left")
    axes[0].set_ylabel("speed (yd/s), mean and IQR")
    fig.suptitle("Figure 1. One yardstick: the same speed-vs-time measurement at the Combine and after the snap", fontsize=11)
    fig.tight_layout(); fig.savefig(OUT / "fig1_profiles.png", dpi=160); plt.close(fig)


def fig2_reliability(g, att):
    rows = []
    for side, outs in [("OL", ["pressure_allowed", "peak_pressure_probability_allowed"]), ("DL", ["player_get_off", "pressured"])]:
        r = split_half_reliability(g[g.side == side], ["nfl_id"], MET + outs, 100)
        r["side"] = side; rows.append(r)
    rel = pd.concat(rows)
    # Combine 40: attempt 1 vs 2, year-centred, linemen only
    f = att[(att.drill_name == "FORTY_YARD_DASH")].copy()
    pos = pd.read_csv(RAW / "players.csv").set_index("nfl_id").nfl_position
    f = f[f.nfl_id.map(pos).isin(["T", "G", "C", "DT", "DE", "OLB", "NT"])]
    f["rank"] = f.groupby("nfl_id").attempt.rank(method="first")
    p1 = f[f["rank"] == 1].set_index("nfl_id"); p2 = f[f["rank"] == 2].set_index("nfl_id"); cm = p1.index.intersection(p2.index)
    c40 = []
    for m in MET:
        a, b = p1.loc[cm, m].astype(float), p2.loc[cm, m].astype(float); yr = p1.loc[cm, "draft_year"]
        a = a - a.groupby(yr).transform("mean"); b = b - b.groupby(yr).transform("mean"); ok = a.notna() & b.notna()
        r1 = np.corrcoef(a[ok], b[ok])[0, 1]
        # r_sb = reliability of the mean of the two attempts (the quantity used downstream)
        c40.append({"metric": m, "n": int(ok.sum()), "r_single": r1, "r_sb": 2 * r1 / (1 + r1)})
    c40 = pd.DataFrame(c40)
    RES["reliability_game"] = rel.round(3).to_dict("records"); RES["reliability_40"] = c40.round(3).to_dict("records")
    fig, ax = plt.subplots(figsize=(8.5, 5.6))
    ms = MET
    y = np.arange(len(ms))
    for off, (lab, df, col) in enumerate([("NFL snaps, OL (odd vs even weeks, n=%d)" % rel[rel.side == "OL"].n.max(), rel[rel.side == "OL"], C_OL),
                                          ("NFL snaps, DL (odd vs even weeks, n=%d)" % rel[rel.side == "DL"].n.max(), rel[rel.side == "DL"], C_DL),
                                          ("Combine 40, linemen, mean of 2 attempts (SB from attempt 1 vs 2, n=%d)" % c40.n.max(), c40, C_COMBINE)]):
        vals = [df.set_index("metric").loc[m, "r_sb"] for m in ms]
        ax.barh(y + (off - 1) * 0.27, vals, height=0.25, color=col, label=lab)
    ax.set_yticks(y); ax.set_yticklabels([LABELS[m] for m in ms]); ax.invert_yaxis(); ax.set_xlim(-0.05, 1.05)
    ax.axvline(0.7, ls="--", color="grey", lw=1); ax.text(0.71, -0.45, "0.70", color="grey", fontsize=8)
    ax.set_xlabel("reliability of the quantity used (split-half with Spearman-Brown for snaps; two-attempt mean for the Combine 40)", fontsize=8.5)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=1, frameon=False); ax.grid(axis="x", alpha=.3)
    ax.set_title("Figure 2. In-game trench movement is a stable signature; Combine starts repeat far less", fontsize=10)
    fig.tight_layout(); fig.savefig(OUT / "fig2_reliability.png", dpi=160); plt.close(fig)


def fig3_transfer(dol, ddl):
    """Forest plot: same metric Combine -> in-game, partial on height, weight, pick."""
    panels = []
    for name, d, src in [("OL kick-slide drill → OL snaps", dol, "c_"), ("OL 40-yd start → OL snaps", dol, "f_"),
                         ("DL pass-rush drill → DL snaps", ddl, "c_"), ("DL 40-yd start → DL snaps", ddl, "f_")]:
        t = pd.concat([corr_table(d, [src + m], ["g_" + m], controls=CTRL, n_boot=2000, seed=1) for m in MET[:6]])
        t["metric"] = MET[:6]; t["panel"] = name; panels.append(t)
    t = pd.concat(panels)
    RES["transfer"] = t.round(3).to_dict("records")
    fig, axes = plt.subplots(1, 4, figsize=(12, 3.6), sharey=True)
    for ax, (name, sub) in zip(axes, t.groupby("panel", sort=False)):
        y = np.arange(len(sub)); col = C_OL if name.startswith("OL") else C_DL
        ax.errorbar(sub.r, y, xerr=[sub.r - sub.lo, sub.hi - sub.r], fmt="o", color=col, capsize=3)
        ax.axvline(0, color="grey", lw=1); ax.set_xlim(-0.6, 0.8); ax.set_title(f"{name}\n(n={int(sub.n.max())})", fontsize=9); ax.grid(axis="x", alpha=.3)
        ax.set_yticks(y); ax.set_yticklabels([LABELS[m] for m in sub.metric])
    axes[0].invert_yaxis(); fig.supxlabel("partial correlation (height, weight, draft slot held fixed), 95% bootstrap CI", fontsize=9)
    fig.suptitle("Figure 3. Does the same movement metric carry from the Combine to NFL snaps?", fontsize=11)
    fig.tight_layout(); fig.savefig(OUT / "fig3_transfer.png", dpi=160); plt.close(fig)


def fig4_ol_scatter(dol):
    d = dol.dropna(subset=["c_s_at_d1", "g_s_at_d1"])
    t = corr_table(d, ["c_s_at_d1"], ["g_s_at_d1"], controls=CTRL, n_boot=2000, seed=2).iloc[0]
    RES["ol_scatter"] = {"n": int(t.n), "r": round(t.r, 3), "lo": round(t.lo, 3), "hi": round(t.hi, 3)}
    fig, ax = plt.subplots(figsize=(6.2, 4.8))
    for pos, mk in [("T", "o"), ("G", "s"), ("C", "^")]:
        s = d[d.nfl_position == pos]; ax.scatter(s.c_s_at_d1, s.g_s_at_d1, marker=mk, s=42, color=C_OL, alpha=.75, label=f"{pos} (n={len(s)})")
    for _, r in pd.concat([d.nlargest(2, "g_s_at_d1"), d.nsmallest(2, "g_s_at_d1")]).iterrows():
        ax.annotate(r.display_name, (r.c_s_at_d1, r.g_s_at_d1), fontsize=7, xytext=(4, 3), textcoords="offset points")
    b = np.polyfit(d.c_s_at_d1, d.g_s_at_d1, 1); xs = np.linspace(d.c_s_at_d1.min(), d.c_s_at_d1.max(), 10); ax.plot(xs, np.polyval(b, xs), color="k", lw=1)
    ax.set_xlabel("Combine kick-slide drill: speed at 1 yd (z within class, mean of L/R)"); ax.set_ylabel("NFL pass-pro snaps: speed at 1 yd (z within position)")
    ax.set_title(f"Figure 4. OL: the kick-slide drill predicts the in-game set\npartial r = {t.r:.2f} (95% CI {t.lo:.2f} to {t.hi:.2f}), n = {int(t.n)}", fontsize=9.5)
    ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(OUT / "fig4_ol_scatter.png", dpi=160); plt.close(fig)


def fig5_dl_outcome(ddl):
    d = ddl.copy(); d["edge"] = d.nfl_position.isin(["DE", "OLB"])
    tg = corr_table(d, ["g_player_get_off"], ["g_pressured"], controls=["combine_weight", "pick"], n_boot=2000, seed=3).iloc[0]
    t2 = corr_table(d, ["g_t_d1_d2"], ["g_pressured"], controls=["combine_weight", "pick"], n_boot=2000, seed=3).iloc[0]
    ts = corr_table(d, ["g_s_at_d1"], ["g_pressured"], controls=["combine_weight", "pick"], n_boot=2000, seed=3).iloc[0]
    RES["dl_outcome"] = {k: {"n": int(v.n), "r": round(v.r, 3), "lo": round(v.lo, 3), "hi": round(v.hi, 3)} for k, v in [("get_off", tg), ("t_d1_d2", t2), ("s_at_d1", ts)]}
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, xcol, tt, lab in [(axes[0], "g_player_get_off", tg, "NGS get-off: time to cross the line (z within position)"),
                              (axes[1], "g_t_d1_d2", t2, "second yard: time from 1 yd to 2 yd of penetration (z within position)")]:
        for e, mk, nm in [(True, "o", "edge (DE/OLB)"), (False, "s", "interior (DT/NT)")]:
            s = d[d.edge == e]; ax.scatter(s[xcol], s.g_pressured, marker=mk, s=42, color=C_DL, alpha=.75, label=nm)
        b = np.polyfit(d[xcol], d.g_pressured, 1); xs = np.linspace(d[xcol].min(), d[xcol].max(), 10); ax.plot(xs, np.polyval(b, xs), color="k", lw=1)
        ax.set_xlabel(lab, fontsize=9); ax.set_title(f"partial r = {tt.r:.2f} (95% CI {tt.lo:.2f} to {tt.hi:.2f}), n = {int(tt.n)}", fontsize=9); ax.grid(alpha=.3)
    axes[0].set_ylabel("pressure rate on pass-rush snaps (z within position)"); axes[0].legend(fontsize=8)
    fig.suptitle("Figure 5. DL: the second yard after the snap predicts pressure better than the first (weight and draft slot held fixed)", fontsize=10)
    fig.tight_layout(); fig.savefig(OUT / "fig5_dl_outcome.png", dpi=160); plt.close(fig)


def fig6_ol_within(g, base):
    ol = g[g.side == "OL"].copy()
    pm = player_means(ol, ["nfl_id"], ["depth_10", "set_angle_10", "t_d1_d2"], 100).set_index("nfl_id")
    x = ol[ol.nfl_id.isin(pm.index) & ol.lined_up_position.isin(["T", "G", "C"])].copy()
    x = x.merge(base[["nfl_id", "pick", "combine_weight"]], on="nfl_id").dropna(subset=["peak_pressure_probability_allowed", "dropback_duration", "pass_rushers_encountered"])
    x = x.dropna(subset=["t_d1_d2"])  # one model sample for both models
    for m in ["depth_10", "set_angle_10", "t_d1_d2"]:
        x["P_" + m] = x.nfl_id.map(pm[m]); x["w_" + m] = x[m] - x.groupby("nfl_id")[m].transform("mean")  # true within-player deviation
    f_between = "peak_pressure_probability_allowed ~ C(lined_up_position) + dropback_duration + pass_rushers_encountered + pick + combine_weight + P_depth_10 + P_set_angle_10 + P_t_d1_d2"
    mb = smf.ols(f_between, data=x).fit(cov_type="cluster", cov_kwds={"groups": x.nfl_id})
    xx = x
    mw = smf.ols("peak_pressure_probability_allowed ~ w_depth_10 + w_set_angle_10 + w_t_d1_d2 + dropback_duration + pass_rushers_encountered", data=xx).fit(cov_type="cluster", cov_kwds={"groups": xx.nfl_id})
    RES["ol_between_model"] = {k: {"coef": round(mb.params[k], 4), "se": round(mb.bse[k], 4), "p": round(mb.pvalues[k], 4)} for k in ["P_depth_10", "P_set_angle_10", "P_t_d1_d2"]}
    RES["ol_within_model"] = {k: {"coef": round(mw.params[k], 4), "se": round(mw.bse[k], 4), "p": round(mw.pvalues[k], 4)} for k in ["w_depth_10", "w_set_angle_10", "w_t_d1_d2"]}
    RES["ol_model_n"] = int(mb.nobs)
    # binned plot: deviation of set depth from the player's own mean vs pressure prob (residual on situation)
    x["resid"] = x.peak_pressure_probability_allowed - smf.ols("peak_pressure_probability_allowed ~ C(lined_up_position) + dropback_duration + pass_rushers_encountered", data=x).fit().fittedvalues
    x["bin"] = pd.qcut(x.w_depth_10, 10, labels=False)
    b = x.groupby("bin").agg(dev=("w_depth_10", "mean"), y=("resid", "mean"), se=("resid", lambda v: v.std() / np.sqrt(len(v))))
    pb = pm.join(x.groupby("nfl_id").resid.mean().rename("y"))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(pb.depth_10, pb.y, color=C_OL, alpha=.75, s=40); axes[0].set_xlabel("player's mean set depth at 1.0 s (yd)"); axes[0].set_ylabel("pressure prob. allowed, net of position & situation")
    axes[0].set_title(f"Between players: no relation (coef p = {mb.pvalues['P_depth_10']:.2f}, n = {len(pb)})", fontsize=9); axes[0].grid(alpha=.3)
    axes[1].errorbar(b.dev, b.y, yerr=1.96 * b.se, fmt="o-", color=C_OL, capsize=3); axes[1].axhline(0, color="grey", lw=1)
    axes[1].set_xlabel("this snap's set depth minus the player's own mean (yd)"); axes[1].set_ylabel("")
    axes[1].set_title(f"Within a player: deeper-than-usual sets are the pressured ones\n(+{mw.params['w_depth_10']:.3f} per yd, n = {int(mw.nobs):,} snaps)", fontsize=9); axes[1].grid(alpha=.3)
    fig.suptitle("Figure 6. OL: set depth is a style, not a grade", fontsize=11)
    fig.tight_layout(); fig.savefig(OUT / "fig6_ol_within.png", dpi=160); plt.close(fig)


def dl_play_level_models(g, base):
    """DL: pressure on a pass-rush snap vs the player's mean get-off / second
    yard, with alignment, blitz, down, distance, draft slot and weight."""
    dl = g[g.side == "DL"].copy()
    pm = player_means(dl, ["nfl_id"], ["t_d1_d2", "s_at_d1", "player_get_off"], 100).set_index("nfl_id")
    x = dl[dl.nfl_id.isin(pm.index)].merge(base[["nfl_id", "pick", "combine_weight", "ten_yd_split"]], on="nfl_id").dropna(subset=["down", "yards_to_go"])
    for m in ["t_d1_d2", "s_at_d1", "player_get_off"]:
        x["P_" + m] = x.nfl_id.map(pm[m])
    x["edge"] = x.lined_up_position.isin(["DE", "OLB"]).astype(float); x["down"] = x.down.astype(int)
    x["blitz"] = x.blitzing.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}).fillna(0.0)
    out = {}
    for name, f in [("get_off", "pressured ~ edge + blitz + C(down) + yards_to_go + pick + combine_weight + P_player_get_off"),
                    ("second_yard", "pressured ~ edge + blitz + C(down) + yards_to_go + pick + combine_weight + P_t_d1_d2"),
                    ("both", "pressured ~ edge + blitz + C(down) + yards_to_go + pick + combine_weight + P_player_get_off + P_t_d1_d2"),
                    ("official_10yd", "pressured ~ edge + blitz + C(down) + yards_to_go + pick + combine_weight + ten_yd_split")]:
        xx = x.dropna(subset=["ten_yd_split"]) if "ten_yd_split" in f else x
        m = smf.ols(f, data=xx).fit(cov_type="cluster", cov_kwds={"groups": xx.nfl_id})
        out[name] = {"n": int(m.nobs), **{k: {"coef": round(m.params[k], 4), "se": round(m.bse[k], 4), "p": round(m.pvalues[k], 4)}
                                        for k in m.params.index if k.startswith("P_") or k == "ten_yd_split"}}
    RES["dl_play_level_models"] = out


def fig7_weekly(g, base):
    dl = g[g.side == "DL"].copy()
    wk = dl.groupby(["nfl_id", "season", "week"]).agg(v=("t_d1_d2", "mean"), n=("t_d1_d2", "size")).reset_index()
    wk = wk[wk.n >= 10]
    pm = player_means(dl, ["nfl_id"], ["t_d1_d2"], 100).set_index("nfl_id")
    game_sd = wk.groupby("nfl_id").v.std().median(); between_sd = pm.t_d1_d2.std()
    RES["weekly"] = {"per_game_sd": round(game_sd, 4), "between_player_sd": round(between_sd, 4), "games_for_r70": int(np.ceil(2.33 * (game_sd ** 2) / (between_sd ** 2)))}
    # pick 3 players with most games, show rolling 4-game mean
    top = wk.groupby("nfl_id").size().nlargest(3).index
    fig, ax = plt.subplots(figsize=(9, 4))
    names = base.set_index("nfl_id").display_name
    for i, pid in enumerate(top):
        s = wk[wk.nfl_id == pid].sort_values(["season", "week"]).reset_index(drop=True)
        xs = np.arange(len(s)); ax.plot(xs, s.v, "o", alpha=.35, color=f"C{i}", ms=4)
        ax.plot(xs, s.v.rolling(4, min_periods=2).mean(), color=f"C{i}", lw=2, label=f"{names[pid]} ({s.season.min()}–{s.season.max()})")
    ax.axhspan(pm.t_d1_d2.mean() - between_sd, pm.t_d1_d2.mean() + between_sd, color="grey", alpha=.12, label="±1 SD between players")
    ax.set_xlabel("game number (regular season, ≥10 rush snaps)"); ax.set_ylabel("time from 1 yd to 2 yd (s), lower = faster")
    ax.set_title(f"Figure 7. Week-to-week monitor of the second yard: one game is as noisy as the between-player spread\n(game SD {game_sd:.3f} s vs between-player SD {between_sd:.3f} s), so the line is a 4-game rolling mean", fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(OUT / "fig7_weekly.png", dpi=160); plt.close(fig)


def main():
    g, att, players, base = load_all()
    dol = player_table(g, att, base, "OL", "PASS_RUSH_DROP", ["pressure_allowed", "peak_pressure_probability_allowed"])
    ddl = player_table(g, att, base, "DL", "PASS_RUSH_DRILL", ["player_get_off", "pressured", "quick_pressure"])
    RES["n"] = {"ol_players": int(len(dol)), "dl_players": int(len(ddl)), "ol_snaps": int((g.side == "OL").sum()), "dl_snaps": int((g.side == "DL").sum()),
                "combine_ol_attempts": int((att.drill_name == "PASS_RUSH_DROP").sum()), "combine_dl_attempts": int((att.drill_name == "PASS_RUSH_DRILL").sum())}
    # official Combine numbers -> in-game (for the text)
    RES["official"] = {}
    for nm, d, ys in [("OL", dol, ["g_s_at_d1", "g_depth_10"]), ("DL", ddl, ["g_t_d1", "g_player_get_off", "g_pressured"])]:
        t = corr_table(d, ["forty", "ten_yd_split", "pick"], ys, controls=CTRL, n_boot=1000, seed=4)
        RES["official"][nm] = t.round(3).to_dict("records")
    RES["combine_onset"] = att[att.drill_name.isin(["PASS_RUSH_DROP", "PASS_RUSH_DRILL", "FORTY_YARD_DASH"])].groupby("drill_name").agg(
        attempts=("event_id", "size"), frac_moving_first_frame=("first_recorded_speed", lambda v: float((v >= 0.5).mean())),
        median_t_shift=("t_shift", "median")).round(3).to_dict("index")
    fig1_profiles(); fig2_reliability(g, att); fig3_transfer(dol, ddl); fig4_ol_scatter(dol); fig5_dl_outcome(ddl); fig6_ol_within(g, base); dl_play_level_models(g, base); fig7_weekly(g, base)
    (OUT.parent / "results.json").write_text(json.dumps(RES, indent=1, default=float), encoding="utf-8")
    print(json.dumps({k: RES[k] for k in ["n", "combine_onset", "ol_scatter", "dl_outcome", "weekly", "ol_between_model", "ol_within_model", "ol_model_n", "dl_play_level_models"]}, indent=1, default=float))


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('wrote src/make_figures.py')

## 1. In-game extraction: first 3 s after `ball_snap` for every OL/DL prospect, regular season only

In [ ]:
from trench_features import *

# ---------------------------------------------------------------------------
# In-game extraction body (appended after trench_features.py by build script)
# ---------------------------------------------------------------------------
import glob
import os
import time as _time
import numpy as np

# competition files may sit one folder deeper than the slug; locate games.csv
BASE = os.path.dirname(glob.glob("/kaggle/input/**/games.csv", recursive=True)[0]) + "/"
print("BASE", BASE)
OL = ["T", "G", "C"]
DL = ["DT", "DE", "OLB", "NT"]
T_LO, T_HI = -0.3, 3.0
NA = ["NA", ""]  # competition CSVs encode nulls as NA

t0 = _time.time()
games = pl.read_csv(BASE + "games.csv", null_values=NA).filter(pl.col("season_type") == "REG")
reg_ids = games["game_id"].to_list()
players = pl.read_csv(BASE + "players.csv", null_values=NA).select("nfl_id", "nfl_position")
trench = players.filter(pl.col("nfl_position").is_in(OL + DL))
trench_ids = trench["nfl_id"].to_list()
print("REG games", len(reg_ids), "trench players", len(trench_ids))

PP_COLS = [
    "game_id", "play_id", "nfl_id", "lined_up_position", "play_direction", "quarter", "down",
    "yards_to_go", "offense_formation", "pass_result", "pass_length", "play_nullified_by_penalty",
    "yards_gained", "expected_points_added", "dropback_duration", "extended_sack_allowed",
    "pass_rushers_encountered", "peak_pressure_probability_allowed", "pressure_allowed",
    "sack_allowed", "time_to_pressure_allowed", "blitzing", "player_get_off", "sack",
    "time_to_pressure", "time_to_qb_hurry", "quick_pressure", "unblocked_pressure",
    "in_motion_at_ball_snap", "team_abbr", "possession_team",
]
pp = pl.read_csv(BASE + "player_play.csv", columns=PP_COLS, infer_schema_length=200000, null_values=NA)
pp = pp.filter(pl.col("nfl_id").is_in(trench_ids) & pl.col("game_id").is_in(reg_ids))
pp = pp.join(trench, on="nfl_id").join(games.select("game_id", "season", "week"), on="game_id")
pp = pp.with_columns(
    pl.when(pl.col("nfl_position").is_in(OL)).then(pl.lit("OL")).otherwise(pl.lit("DL")).alias("side")
)
# pass-pro snaps for OL, pass-rush snaps for DL
pp = pp.filter(
    ((pl.col("side") == "OL") & pl.col("dropback_duration").is_not_null())
    | ((pl.col("side") == "DL") & pl.col("player_get_off").is_not_null())
)
print("player_play trench pass rows", pp.shape, pp.group_by("side").len())

# facing target: OL faces the defense (+x when play goes right); DL faces offense
pp = pp.with_columns(
    pl.when(pl.col("play_direction") == "right").then(-1.0).otherwise(1.0).alias("ux"),
    pl.lit(0.0).alias("uy"),
    pl.when(pl.col("play_direction") == "right")
    .then(pl.when(pl.col("side") == "OL").then(90.0).otherwise(270.0))
    .otherwise(pl.when(pl.col("side") == "OL").then(270.0).otherwise(90.0))
    .alias("face"),
)
key = ["game_id", "play_id", "nfl_id"]
pp_axis = pp.select(key + ["ux", "uy", "face"])

parts = []
profiles = []
diag = []
for yr in (2023, 2024, 2025):
    t1 = _time.time()
    lf = pl.scan_csv(
        BASE + f"game_tracking_{yr}.csv",
        schema_overrides={"event": pl.String, "time": pl.String}, null_values=NA,
    )
    lf = lf.filter(pl.col("nfl_id").is_in(trench_ids) & pl.col("game_id").is_in(reg_ids))
    lf = lf.with_columns(pl.col("time").str.to_datetime("%Y-%m-%dT%H:%M:%S%.f").alias("ts"))
    snap = (
        lf.filter(pl.col("event") == "ball_snap")
        .group_by("game_id", "play_id")
        .agg(pl.col("ts").min().alias("t_snap"))
    )
    df = (
        lf.join(snap, on=["game_id", "play_id"], how="inner")
        .with_columns(((pl.col("ts") - pl.col("t_snap")).dt.total_milliseconds() / 1000.0).alias("t"))
        .filter((pl.col("t") >= T_LO) & (pl.col("t") <= T_HI))
        .select(key + ["t", "x", "y", "s", "a", "o", "dir", "event"])
        .collect()
    )
    n_plays_snap = df.select("game_id", "play_id").unique().height
    df = df.join(pp_axis, on=key, how="inner").sort(key + ["t"])
    df = add_depth_lateral(df, key)
    feats = window_features(df, key)
    profiles.append(speed_profiles(df, key))
    # dominant movement direction in first 1 s (deg, dir column) for set-type work
    first_dir = (
        df.filter((pl.col("t") > 0.3) & (pl.col("t") <= 1.0))
        .group_by(key).agg(pl.col("dir").median().alias("dir_med_03_10"))
    )
    feats = feats.join(first_dir, on=key, how="left")
    parts.append(feats)
    diag.append({"season": yr, "plays_with_snap": n_plays_snap,
                 "player_plays": feats.height, "sec": round(_time.time() - t1, 1)})
    print(diag[-1], flush=True)

feats = pl.concat(parts)
out = pp.drop(["ux", "uy", "face"]).join(feats, on=key, how="inner")
out.write_csv("/kaggle/working/trench_game_features.csv")
pl.concat(profiles).join(pp.select(key + ["side", "lined_up_position", "season", "week"]), on=key, how="inner").write_parquet("/kaggle/working/trench_game_profiles.parquet")
pl.DataFrame(diag).write_csv("/kaggle/working/extract_diagnostics.csv")
print("OUT", out.shape, "elapsed", round(_time.time() - t0, 1), "s")
print(out.group_by("side", "nfl_position").len().sort("side", "nfl_position"))
print("pre-snap speed quantiles", out["pre_s"].quantile(0.5), out["pre_s"].quantile(0.95))
dl = out.filter(pl.col("side") == "DL").drop_nulls(["t_d1", "player_get_off"])
if dl.height:
    r = np.corrcoef(dl["t_d1"].to_numpy(), dl["player_get_off"].to_numpy())[0, 1]
    print("DL t_d1 vs player_get_off play-level r =", round(float(r), 3), "n =", dl.height)
print(out.select(["depth_10", "lat_abs_10", "s_10", "smax_15", "t_s2", "t_d1", "orel_10", "set_angle_10"]).describe())

for f in ['trench_game_features.csv', 'trench_game_profiles.parquet', 'extract_diagnostics.csv']:
    shutil.move(f, 'data/processed/' + f)

## 2. Combine drill metrics with the same function

In [ ]:
import build_combine_trench_metrics
build_combine_trench_metrics.main()

## 3. Reliability, transfer, outcome models and figures

In [ ]:
import make_figures
make_figures.main()

In [ ]:
from IPython.display import Image, display
for i in range(1, 8):
    display(Image(filename=sorted(glob.glob(f'output/figures/fig{i}_*.png'))[0]))

## 4. All numbers used in the writeup

In [ ]:
import json
print(json.dumps(json.load(open('output/results.json')), indent=1))